### Import libs

In [1]:
# Imports

!python -m pip install matplotlib pandas numpy
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


### Load dataset

In [2]:
# Load in dataset

df = pd.read_csv("cleaned_bank_fraud.csv", header=None, na_values="?")

# Metadata about the data frame
print(f"Shape: {df.shape}")
print(f"Head: {df.head()}")
print(f"Data types: {df.dtypes.value_counts()}")

Shape: (50001, 25)
Head:     0            1                                2                 3   \
0  NaN  Customer_ID  Transaction_Amount (in Million)  Transaction_Time   
1  0.0      24239.0                              6.0             10:54   
2  1.0      77250.0                              9.0             19:23   
3  2.0      34294.0                              3.0             10:20   
4  3.0      92041.0                              1.0             14:11   

                 4                 5            6                  7   \
0  Transaction_Date  Transaction_Type  Merchant_ID  Merchant_Category   
1        2025-03-08                 2      97028.0                  0   
2        2025-01-17                 0      27515.0                  0   
3        2025-04-30                 2      13810.0                  2   
4        2025-02-21                 1      10501.0                  4   

                     8                       9   ...  \
0  Transaction_Location  Customer_H

/var/folders/2g/hrmk7dfj1tsgx15lx33s7wsm0000gn/T/ipykernel_2444/3148510279.py:3: DtypeWarning: Columns (1,2,5,6,7,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("cleaned_bank_fraud.csv", header=None, na_values="?")


### Importing Original Data Set and Finding Missing Cells

In [ ]:
raw_df = pd.read_csv("original_dataset/bank_fraud.csv")

print(f"Raw shape: {raw_df.shape}")
print(f"Cleaned shape: {df.shape}")

print(raw_df.columns.tolist())
print(raw_df.isna().sum().sum(), "missing cells")

Raw shape: (50000, 25)
Cleaned shape: (50000, 25)
['Transaction_ID', 'Customer_ID', 'Transaction_Amount (in Million)', 'Transaction_Time', 'Transaction_Date', 'Transaction_Type', 'Merchant_ID', 'Merchant_Category', 'Transaction_Location', 'Customer_Home_Location', 'Distance_From_Home', 'Device_ID', 'IP_Address', 'Card_Type', 'Account_Balance (in Million)', 'Daily_Transaction_Count', 'Weekly_Transaction_Count', 'Avg_Transaction_Amount (in Million)', 'Max_Transaction_Last_24h (in Million)', 'Is_International_Transaction', 'Is_New_Merchant', 'Failed_Transaction_Count', 'Unusual_Time_Transaction', 'Previous_Fraud_Count', 'Fraud_Label']
150 missing cells


### 2.1 Processing 

In [7]:
from sklearn.model_selection import train_test_split

TARGET = "Fraud_Label"   

data = raw_df.dropna(subset=[TARGET]).copy()
data = data.drop(columns=["Transaction_ID", "Customer_ID"])  


for col in data.select_dtypes(include="object").columns:
    codes, _ = pd.factorize(data[col])
    data[col] = codes
    data.loc[data[col] == -1, col] = np.nan   

sample = data.groupby(TARGET, group_keys=False).apply(
    lambda g: g.sample(frac=5000/len(data), random_state=42))

X = sample.drop(columns=[TARGET]).values
y = sample[TARGET].values.astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)
print(X_train.shape, np.bincount(y_train))

(4000, 22) [3806  194]


/var/folders/2g/hrmk7dfj1tsgx15lx33s7wsm0000gn/T/ipykernel_2444/3930555915.py:14: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sample = data.groupby(TARGET, group_keys=False).apply(


### Running eLCS



In [9]:
from skeLCS import eLCS
from sklearn.metrics import balanced_accuracy_score, f1_score, recall_score, confusion_matrix

model = eLCS(learning_iterations=20000, N=1000)
model.fit(X_train, y_train)
pred = model.predict(X_test)

print("Balanced acc:", balanced_accuracy_score(y_test, pred))
print("F1:", f1_score(y_test, pred))
print("Recall:", recall_score(y_test, pred))
print(confusion_matrix(y_test, pred))

Balanced acc: 0.5
F1: 0.0
Recall: 0.0
[[952   0]
 [ 48   0]]
